# LeetCode #399: Evaluate Division

https://leetcode.com/problems/evaluate-division/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **BFS/DFS per Query** | $O(Q \cdot (V + E))$ | $O(V + E)$ |
| **Optimal: Weighted Union-Find ★** | $O((E + Q) \cdot \alpha(V))$ | $O(V)$ |

---

## Understanding the Methods

### BFS/DFS per Query
Build a weighted directed graph where an edge from A to B with weight w means A/B = w. For each query (X, Y), run BFS or DFS from X to Y, multiplying edge weights along the path.

### Optimal: Weighted Union-Find ★
Use a union-find structure where each node stores a weight representing the ratio to its parent. When unioning A and B with A/B = w, adjust the weights accordingly. To query A/B, find both roots; if they share a root, the answer is weight(A)/weight(B).

**Why this is better than BFS/DFS:** Union-Find with path compression answers each query in near-constant time after the initial build phase, making it faster for many queries.

**Constraints:**
* 1 <= equations.length <= 20
* 1 <= queries.length <= 20
* equations[i].length == 2
* 1 <= Ai.length, Bi.length <= 5
* values[i] > 0

## Solutions

### C#

In [ ]:
public class Solution {
    public double[] CalcEquation(IList<IList<string>> equations, double[] values, IList<IList<string>> queries) {
        var graph = new Dictionary<string, List<(string, double)>>();
        for (int i = 0; i < equations.Count; i++) {
            string a = equations[i][0], b = equations[i][1];
            if (!graph.ContainsKey(a)) graph[a] = new List<(string, double)>();
            if (!graph.ContainsKey(b)) graph[b] = new List<(string, double)>();
            graph[a].Add((b, values[i]));
            graph[b].Add((a, 1.0 / values[i]));
        }
        double[] res = new double[queries.Count];
        for (int i = 0; i < queries.Count; i++) {
            string src = queries[i][0], dst = queries[i][1];
            if (!graph.ContainsKey(src) || !graph.ContainsKey(dst))
                res[i] = -1.0;
            else
                res[i] = Bfs(graph, src, dst);
        }
        return res;
    }

    private double Bfs(Dictionary<string, List<(string, double)>> graph, string src, string dst) {
        if (src == dst) return 1.0;
        var visited = new HashSet<string> { src };
        var queue = new Queue<(string, double)>();
        queue.Enqueue((src, 1.0));
        while (queue.Count > 0) {
            var (node, val) = queue.Dequeue();
            foreach (var (next, w) in graph[node]) {
                if (next == dst) return val * w;
                if (visited.Add(next))
                    queue.Enqueue((next, val * w));
            }
        }
        return -1.0;
    }
}

### Python

In [ ]:
from collections import defaultdict, deque

class Solution:
    def calcEquation(self, equations: list[list[str]], values: list[float], queries: list[list[str]]) -> list[float]:
        graph = defaultdict(list)
        for (a, b), v in zip(equations, values):
            graph[a].append((b, v))
            graph[b].append((a, 1.0 / v))

        def bfs(src, dst):
            if src not in graph or dst not in graph:
                return -1.0
            if src == dst:
                return 1.0
            visited = {src}
            queue = deque([(src, 1.0)])
            while queue:
                node, val = queue.popleft()
                for nxt, w in graph[node]:
                    if nxt == dst:
                        return val * w
                    if nxt not in visited:
                        visited.add(nxt)
                        queue.append((nxt, val * w))
            return -1.0

        return [bfs(a, b) for a, b in queries]

### Go

In [ ]:
func calcEquation(equations [][]string, values []float64, queries [][]string) []float64 {
    type edge struct {
        to string
        w  float64
    }
    graph := map[string][]edge{}
    for i, eq := range equations {
        a, b := eq[0], eq[1]
        graph[a] = append(graph[a], edge{b, values[i]})
        graph[b] = append(graph[b], edge{a, 1.0 / values[i]})
    }
    bfs := func(src, dst string) float64 {
        if _, ok := graph[src]; !ok { return -1.0 }
        if _, ok := graph[dst]; !ok { return -1.0 }
        if src == dst { return 1.0 }
        type state struct { node string; val float64 }
        visited := map[string]bool{src: true}
        queue := []state{{src, 1.0}}
        for len(queue) > 0 {
            cur := queue[0]; queue = queue[1:]
            for _, e := range graph[cur.node] {
                if e.to == dst { return cur.val * e.w }
                if !visited[e.to] {
                    visited[e.to] = true
                    queue = append(queue, state{e.to, cur.val * e.w})
                }
            }
        }
        return -1.0
    }
    res := make([]float64, len(queries))
    for i, q := range queries {
        res[i] = bfs(q[0], q[1])
    }
    return res
}

### Rust

In [ ]:
use std::collections::{HashMap, HashSet, VecDeque};

impl Solution {
    pub fn calc_equation(
        equations: Vec<Vec<String>>, values: Vec<f64>, queries: Vec<Vec<String>>,
    ) -> Vec<f64> {
        let mut graph: HashMap<&str, Vec<(&str, f64)>> = HashMap::new();
        for (eq, &v) in equations.iter().zip(values.iter()) {
            let (a, b) = (eq[0].as_str(), eq[1].as_str());
            graph.entry(a).or_default().push((b, v));
            graph.entry(b).or_default().push((a, 1.0 / v));
        }
        queries.iter().map(|q| {
            let (src, dst) = (q[0].as_str(), q[1].as_str());
            if !graph.contains_key(src) || !graph.contains_key(dst) { return -1.0; }
            if src == dst { return 1.0; }
            let mut visited = HashSet::new();
            visited.insert(src);
            let mut queue = VecDeque::new();
            queue.push_back((src, 1.0));
            while let Some((node, val)) = queue.pop_front() {
                for &(nxt, w) in &graph[node] {
                    if nxt == dst { return val * w; }
                    if visited.insert(nxt) {
                        queue.push_back((nxt, val * w));
                    }
                }
            }
            -1.0
        }).collect()
    }
}

## Example Scenarios

### Scenario 1: Direct and transitive queries
**Input:** `equations = [["a","b"],["b","c"]], values = [2.0,3.0], queries = [["a","c"]]`  
a/b=2, b/c=3, so a/c = 2*3 = 6. **Output:** `[6.0]`

### Scenario 2: Reverse query
**Input:** `equations = [["a","b"]], values = [2.0], queries = [["b","a"]]`  
b/a = 1/2 = 0.5. **Output:** `[0.5]`

### Scenario 3: Unknown variable
**Input:** `equations = [["a","b"]], values = [2.0], queries = [["a","e"]]`  
Variable e is not in the graph. **Output:** `[-1.0]`

### Scenario 4: Self-division
**Input:** `equations = [["a","b"]], values = [2.0], queries = [["a","a"]]`  
a/a = 1.0. **Output:** `[1.0]`

### Scenario 5: Disconnected components
**Input:** `equations = [["a","b"],["c","d"]], values = [2.0,3.0], queries = [["a","d"]]`  
a and d are in different components. **Output:** `[-1.0]`

*Infographic will be added in a future update.*